In [34]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from pydantic import BaseModel, Field
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langgraph.checkpoint.memory import InMemorySaver

In [35]:
load_dotenv()

True

In [36]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model = ChatHuggingFace(llm = llm)


In [37]:
class Jokestate(TypedDict):
    topic: str
    joke:str
    explanation : dict

In [38]:
class Theory(BaseModel):
    points : List[str] = Field(description="5 points of explanantion")
    
  

In [39]:
parser = PydanticOutputParser(pydantic_object=Theory)

In [40]:
def generate_joke(state: Jokestate):
    prompt = f"generate a joke on the topic {state['topic']}"
    result = model.invoke(prompt).content

    return {
        'joke' : result
    }

In [41]:
def explanation (state : Jokestate):
    prompt = f"""write 5 points about the joke : {state['joke']}
    {parser.get_format_instructions()}"""

    result = model.invoke(prompt)

    parsed = parser.parse(result.content)

    return{
        'explanation' : parsed.model_dump()
    }

In [42]:
graph = StateGraph(Jokestate)

graph.add_node('generate_joke', generate_joke)
graph.add_node('explanation' , explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke" , "explanation")
graph.add_edge("explanation" , END)

pointer = InMemorySaver()

workflow = graph.compile(checkpointer=pointer)

In [44]:
config1 = {"configurable": {"thread_id" :"1"}}
workflow.invoke({"topic" : "girl"} , config = config1)

{'topic': 'girl',
 'joke': 'Why did the girl bring a ladder to the party? \n\nBecause she heard the drinks were on the house.',
 'explanation': {'points': ["The joke relies on a play on words, 'on the house' having a double meaning of both free drinks and being on a ladder.",
   'The punchline requires some level of wordplay understanding and the ability to make a connection between two seemingly unrelated concepts.',
   "The joke's effectiveness lies in the unexpected twist on the phrase 'on the house', which is a common phrase used in customer service.",
   "This type of joke is often referred to as a 'play on words' or 'double meaning' joke, as it depends on the listener being familiar with the phrase 'on the house'.",
   "The joke's simplicity and quick setup allow for a quick and effective punchline delivery, making it more likely to elicit a laugh from the audience."]}}

In [45]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'girl', 'joke': 'Why did the girl bring a ladder to the party? \n\nBecause she heard the drinks were on the house.', 'explanation': {'points': ["The joke relies on a play on words, 'on the house' having a double meaning of both free drinks and being on a ladder.", 'The punchline requires some level of wordplay understanding and the ability to make a connection between two seemingly unrelated concepts.', "The joke's effectiveness lies in the unexpected twist on the phrase 'on the house', which is a common phrase used in customer service.", "This type of joke is often referred to as a 'play on words' or 'double meaning' joke, as it depends on the listener being familiar with the phrase 'on the house'.", "The joke's simplicity and quick setup allow for a quick and effective punchline delivery, making it more likely to elicit a laugh from the audience."]}}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10c7d4-f3b